# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df=pd.read_csv("../data/day-of-week-not-scaled.csv")
previous_df=pd.read_csv("../data/dayofweek.csv")
df['dayofweek']=previous_df['dayofweek']

X=df.drop('dayofweek', axis=1)
y=df['dayofweek']

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
X_train, X_valid, y_train, y_valid=train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [3]:
svc = SVC(random_state=21,kernel='rbf',gamma='auto',class_weight=None,C=10,probability=True)
svc.fit(X_train, y_train)
pred = svc.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.87778
precision is 0.88162
recall is: 0.87778


In [4]:
dt = DecisionTreeClassifier(random_state=21,class_weight='balanced',criterion='gini',max_depth=21)
dt.fit(X_train, y_train)
pred = dt.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.86667
precision is 0.8717
recall is: 0.86667


In [5]:
rf = RandomForestClassifier(random_state=21, n_estimators=100, max_depth=24, class_weight='balanced', criterion='entropy')
rf.fit(X_train, y_train)
pred = rf.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.8963
precision is 0.89698
recall is: 0.8963


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [6]:
voting_clf = VotingClassifier(estimators=[('svc', svc), ('dt', dt), ('rf', rf)], voting='hard')
voting_clf.fit(X_train, y_train)
pred = voting_clf.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.9
precision is 0.89993
recall is: 0.9


In [7]:
weights_variants = [
    [1, 1, 1],
    
    [2, 1, 1], 
    [3, 1, 1],
    
    [1, 2, 1],
    [1, 3, 1],
    
    [1, 1, 2],
    [1, 1, 3],
    

    [2, 1, 3], [3, 1, 2],
    [1, 3, 2], [1, 2, 3], 
    [2, 3, 1], [3, 2, 1],
    
    [2, 2, 1],
    [2, 1, 2],
    [1, 2, 2],

    [4, 1, 4]
]

best_accuracy = 0
best_precision = 0
best_voting_clf = None

for weights in weights_variants:
    voting_clf = VotingClassifier(
        estimators=[('svc', svc), ('dt', dt), ('rf', rf)],
        voting='soft',
        weights=weights
    )
    
    voting_clf.fit(X_train, y_train)
    pred = voting_clf.predict(X_valid)
    
    accuracy = accuracy_score(y_valid, pred)
    precision = precision_score(y_valid, pred, average='weighted')
    recall = recall_score(y_valid, pred, average='weighted')
    
    print(f"Веса {weights}:")
    print(f'Accuracy: {accuracy:.5f}')
    print(f'Precision: {precision:.5f}')
    print(f'Recall: {recall:.5f}\n')
    
    if (accuracy > best_accuracy) or \
       (accuracy == best_accuracy and precision > best_precision):
        best_accuracy = accuracy
        best_precision = precision
        best_voting_clf = voting_clf

Веса [1, 1, 1]:
Accuracy: 0.88519
Precision: 0.88840
Recall: 0.88519

Веса [2, 1, 1]:
Accuracy: 0.90000
Precision: 0.90072
Recall: 0.90000

Веса [3, 1, 1]:
Accuracy: 0.90370
Precision: 0.90607
Recall: 0.90370

Веса [1, 2, 1]:
Accuracy: 0.86667
Precision: 0.87170
Recall: 0.86667

Веса [1, 3, 1]:
Accuracy: 0.86667
Precision: 0.87170
Recall: 0.86667

Веса [1, 1, 2]:
Accuracy: 0.90000
Precision: 0.90180
Recall: 0.90000

Веса [1, 1, 3]:
Accuracy: 0.89630
Precision: 0.89780
Recall: 0.89630

Веса [2, 1, 3]:
Accuracy: 0.90000
Precision: 0.89985
Recall: 0.90000

Веса [3, 1, 2]:
Accuracy: 0.90370
Precision: 0.90397
Recall: 0.90370

Веса [1, 3, 2]:
Accuracy: 0.86667
Precision: 0.87170
Recall: 0.86667

Веса [1, 2, 3]:
Accuracy: 0.87778
Precision: 0.88161
Recall: 0.87778

Веса [2, 3, 1]:
Accuracy: 0.86667
Precision: 0.87170
Recall: 0.86667

Веса [3, 2, 1]:
Accuracy: 0.90000
Precision: 0.90240
Recall: 0.90000

Веса [2, 2, 1]:
Accuracy: 0.87037
Precision: 0.87480
Recall: 0.87037

Веса [2, 1, 2]:
Accu

In [8]:
test_pred = best_voting_clf.predict(X_test)
    
test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, average='weighted')
test_recall = recall_score(y_test, test_pred, average='weighted')

print("Результаты на test set:")
print(f'Accuracy: {test_accuracy:.5f}')
print(f'Precision: {test_precision:.5f}')
print(f'Recall: {test_recall:.5f}')

Результаты на test set:
Accuracy: 0.90533
Precision: 0.90881
Recall: 0.90533


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [10]:
bagging_clf = BaggingClassifier(
    base_estimator=svc,
    random_state=21,
    bootstrap=True
)

bagging_clf.fit(X_train, y_train)
pred = bagging_clf.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.88519
precision is 0.89427
recall is: 0.88519


In [12]:
params_list = [
    {'n_estimators': 10, 'max_samples': 0.6},
    {'n_estimators': 20, 'max_samples': 0.7},
    {'n_estimators': 15, 'max_samples': 0.9},
    {'n_estimators': 50, 'max_samples': 0.9},
]

best_accuracy = 0
best_precision = 0
best_bagging_clf = None


for params in params_list:
    bagging_clf = BaggingClassifier(
        base_estimator=svc,
        random_state=21,
        bootstrap=True,
        n_estimators=params['n_estimators'],
        max_samples=params['max_samples']
    )
    
    bagging_clf.fit(X_train, y_train)
    pred = bagging_clf.predict(X_valid)
    
    accuracy = accuracy_score(y_valid, pred)
    precision = precision_score(y_valid, pred, average='weighted')
    recall = recall_score(y_valid, pred, average='weighted')
    
    print(f"Параметры {params}:")
    print(f'Accuracy: {accuracy:.5f}')
    print(f'Precision: {precision:.5f}')
    print(f'Recall: {recall:.5f}\n')
    
    if (accuracy > best_accuracy) or \
       (accuracy == best_accuracy and precision > best_precision):
        best_accuracy = accuracy
        best_precision = precision
        best_bagging_clf = bagging_clf

Параметры {'n_estimators': 10, 'max_samples': 0.6}:
Accuracy: 0.84815
Precision: 0.86501
Recall: 0.84815

Параметры {'n_estimators': 20, 'max_samples': 0.7}:
Accuracy: 0.85185
Precision: 0.85948
Recall: 0.85185

Параметры {'n_estimators': 15, 'max_samples': 0.9}:
Accuracy: 0.87407
Precision: 0.88314
Recall: 0.87407

Параметры {'n_estimators': 50, 'max_samples': 0.9}:
Accuracy: 0.87407
Precision: 0.88390
Recall: 0.87407



In [13]:
test_pred = best_bagging_clf.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, average='weighted')
test_recall = recall_score(y_test, test_pred, average='weighted')

print(f'Accuracy: {test_accuracy:.5f}')
print(f'Precision: {test_precision:.5f}')
print(f'Recall: {test_recall:.5f}')

Accuracy: 0.86686
Precision: 0.87297
Recall: 0.86686


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [14]:
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_options = [True, False]

best_accuracy = 0
best_precision = 0
best_stacking_clf = None


for n_splits in n_splits_list:
    for passthrough in passthrough_options:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        
        stacking_clf = StackingClassifier(
            estimators=[('svc', svc),('dt', dt),('rf', rf)],
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=cv,
            passthrough=passthrough
        )
        
        stacking_clf.fit(X_train, y_train)
        pred = stacking_clf.predict(X_valid)

        accuracy = accuracy_score(y_valid, pred)
        precision = precision_score(y_valid, pred, average='weighted')
        recall = recall_score(y_valid, pred, average='weighted')
        
        print(f"Параметры (n_splits={n_splits}, passthrough={passthrough}):")
        print(f'Accuracy: {accuracy:.5f}')
        print(f'Precision: {precision:.5f}')
        print(f'Recall: {recall:.5f}\n')
        
        if (accuracy > best_accuracy) or \
           (accuracy == best_accuracy and precision > best_precision):
            best_accuracy = accuracy
            best_precision = precision
            best_stacking_clf = stacking_clf

Параметры (n_splits=2, passthrough=True):
Accuracy: 0.90370
Precision: 0.90508
Recall: 0.90370

Параметры (n_splits=2, passthrough=False):
Accuracy: 0.89630
Precision: 0.89678
Recall: 0.89630

Параметры (n_splits=3, passthrough=True):
Accuracy: 0.90370
Precision: 0.90632
Recall: 0.90370

Параметры (n_splits=3, passthrough=False):
Accuracy: 0.89630
Precision: 0.89759
Recall: 0.89630

Параметры (n_splits=4, passthrough=True):
Accuracy: 0.91111
Precision: 0.91327
Recall: 0.91111

Параметры (n_splits=4, passthrough=False):
Accuracy: 0.90370
Precision: 0.90570
Recall: 0.90370

Параметры (n_splits=5, passthrough=True):
Accuracy: 0.90000
Precision: 0.90217
Recall: 0.90000

Параметры (n_splits=5, passthrough=False):
Accuracy: 0.90000
Precision: 0.90056
Recall: 0.90000

Параметры (n_splits=6, passthrough=True):
Accuracy: 0.90370
Precision: 0.90450
Recall: 0.90370

Параметры (n_splits=6, passthrough=False):
Accuracy: 0.90370
Precision: 0.90436
Recall: 0.90370

Параметры (n_splits=7, passthrough=

In [15]:
test_pred = best_stacking_clf.predict(X_test)
    
test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, average='weighted')
test_recall = recall_score(y_test, test_pred, average='weighted')
   
print("Результаты на test set:")
print(f'Accuracy: {test_accuracy:.5f}')
print(f'Precision: {test_precision:.5f}')
print(f'Recall: {test_recall:.5f}')

Результаты на test set:
Accuracy: 0.90533
Precision: 0.90844
Recall: 0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [16]:
best=best_voting_clf

In [17]:
df_analysis = df.copy()
df_analysis['preds'] = best.predict(X)

df_analysis['is_error'] = (df_analysis['dayofweek'] != df_analysis['preds']).astype(int)

def get_error_percentage(group_col_prefix, original_df):
    results = {}
    
    cols = [c for c in original_df.columns if c.startswith(group_col_prefix)]
    
    if group_col_prefix == 'dayofweek':
        grouped = original_df.groupby('dayofweek')['is_error'].mean() * 100
        return grouped.sort_values(ascending=False)

    for col in cols:
        subset = original_df[original_df[col] == 1.0]
        if len(subset) > 0:
            error_perc = subset['is_error'].mean() * 100
            results[col] = error_perc
            
    return pd.Series(results).sort_values(ascending=False)

print(get_error_percentage('dayofweek', df_analysis))

print("\nlabname")
print(get_error_percentage('labname', df_analysis).head())

print("\nusers")
print(get_error_percentage('uid_user', df_analysis).head())

dayofweek
0    9.558824
2    5.369128
5    4.797048
1    4.014599
6    3.089888
4    2.884615
3    1.767677
Name: is_error, dtype: float64

labname
labname_lab03      100.000000
labname_lab05s      11.111111
labname_laba04s      9.615385
labname_laba06s      6.557377
labname_laba06       6.250000
dtype: float64

users
uid_user_22    28.571429
uid_user_6     25.000000
uid_user_23    25.000000
uid_user_17    14.705882
uid_user_16     6.250000
dtype: float64


In [18]:
joblib.dump(best, 'best.joblib')

['best.joblib']